<a href="https://colab.research.google.com/github/masteryongsa82/Portfolio/blob/main/Cipher_Version3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# FULL CRYPTO TOOL (ALL-IN-ONE)
# =========================

# 설치
try:
    from Crypto.Cipher import AES, PKCS1_OAEP
except:
    !pip install pycryptodome

from Crypto.PublicKey import RSA
from Crypto.Cipher import AES, PKCS1_OAEP
from Crypto.Random import get_random_bytes
import base64, json, os, hashlib

# =========================
# 파일 경로
# =========================
PRIVATE_KEY_FILE = "private.pem"
PUBLIC_KEY_FILE = "public.pem"

# =========================
# 키 관련 함수
# =========================
def keys_exist():
    return os.path.exists(PRIVATE_KEY_FILE) and os.path.exists(PUBLIC_KEY_FILE)

def generate_and_save_keys():
    print("🔑 새 키 생성 중...")
    key = RSA.generate(2048)

    with open(PRIVATE_KEY_FILE, "wb") as f:
        f.write(key.export_key())

    with open(PUBLIC_KEY_FILE, "wb") as f:
        f.write(key.publickey().export_key())

    print("키 저장 완료")

def load_keys():
    with open(PRIVATE_KEY_FILE, "rb") as f:
        private_key = RSA.import_key(f.read())

    with open(PUBLIC_KEY_FILE, "rb") as f:
        public_key = RSA.import_key(f.read())

    return private_key, public_key

def show_public_key(public_key):
    print("\n===== 공개키 =====\n")
    print(public_key.export_key().decode())

def show_private_key(private_key):
    print("\n===== 개인키 (절대 공유 금지) =====\n")
    print(private_key.export_key().decode())

def show_fingerprint(public_key):
    fp = hashlib.sha256(public_key.export_key()).hexdigest()
    print("\n===== 키 지문 =====")
    print(fp)

# =========================
# 암호화
# =========================
def encrypt_message(message: str, public_key):
    aes_key = get_random_bytes(32)

    cipher = AES.new(aes_key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(message.encode())

    rsa_cipher = PKCS1_OAEP.new(public_key)
    enc_key = rsa_cipher.encrypt(aes_key)

    data = {
        "k": base64.b64encode(enc_key).decode(),
        "n": base64.b64encode(cipher.nonce).decode(),
        "c": base64.b64encode(ciphertext).decode(),
        "t": base64.b64encode(tag).decode()
    }

    return base64.b64encode(json.dumps(data).encode()).decode()

# =========================
# 복호화
# =========================
def decrypt_message(blob: str, private_key):
    try:
        raw = base64.b64decode(blob)
        data = json.loads(raw)

        enc_key = base64.b64decode(data["k"])
        nonce = base64.b64decode(data["n"])
        ciphertext = base64.b64decode(data["c"])
        tag = base64.b64decode(data["t"])

        rsa_cipher = PKCS1_OAEP.new(private_key)
        aes_key = rsa_cipher.decrypt(enc_key)

        cipher = AES.new(aes_key, AES.MODE_GCM, nonce=nonce)
        plaintext = cipher.decrypt_and_verify(ciphertext, tag)

        return plaintext.decode()
    except:
        return "복호화 실패"

# =========================
# 초기 키 처리
# =========================
print("===== 키 상태 =====")

if keys_exist():
    print("기존 키 발견 → 불러오는 중...")
else:
    print("키 없음 → 새로 생성")
    generate_and_save_keys()

private_key, public_key = load_keys()
print("준비 완료\n")

# =========================
# 메뉴
# =========================
while True:
    print("\n===== 메뉴 =====")
    print("1. 암호화")
    print("2. 복호화")
    print("3. 공개키 보기")
    print("4. 키 지문 보기")
    print("5. 개인키 보기 ")
    print("6. 키 재생성")
    print("7. 종료")

    choice = input("선택: ")

    if choice == "1":
        msg = input("메시지: ")
        print("\n결과:\n")
        print(encrypt_message(msg, public_key))

    elif choice == "2":
        blob = input("암호문: ")
        print("\n결과:\n")
        print(decrypt_message(blob, private_key))

    elif choice == "3":
        show_public_key(public_key)

    elif choice == "4":
        show_fingerprint(public_key)

    elif choice == "5":
        show_private_key(private_key)

    elif choice == "6":
        generate_and_save_keys()
        private_key, public_key = load_keys()
        print("새 키 적용 완료")

    elif choice == "7":
        print("종료")
        break

    else:
        print("다시 선택")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 18.1 MB/s eta 0:00:00
===== 키 상태 =====
키 없음 → 새로 생성
🔑 새 키 생성 중...
키 저장 완료
준비 완료


===== 메뉴 =====
1. 암호화
2. 복호화
3. 공개키 보기
4. 키 지문 보기
5. 개인키 보기 
6. 키 재생성
7. 종료
선택: 7
종료
